In [13]:
%run memclean.py

In [20]:
memclean()

[memclean] Old model removed from RAM. Disk cache untouched — no re-download will occur.


In [28]:
# Step 0 (One-time only): Log into Hugging Face to access gated (restricted) models
from huggingface_hub import login
import os

# Retrieve the Hugging Face access token from ENV
# See https://huggingface.co/settings/tokens
token = os.getenv("HF_TOKEN")

# Log in with the token (needed for gated/restricted models on Hugging Face)
login(token=token)

# Step 1: Import the Hugging Face Transformers library
from transformers import AutoTokenizer, AutoModelForCausalLM

# Step 2: Pick a sample LLM, make sure you've got access to the gated model of your choice
# See https://huggingface.co/settings/gated-repos
#
# model_id = "facebook/opt-125m"
# model_id = "google/gemma-2-2b-it"
# model_id = "meta-llama/Llama-3.2-1B"
model_id = "unsloth/Llama-3.2-1B"

# Step 3: Load a tokenizer and a model
# Only load the model if it isn't already in RAM.
# If it's already loaded, reuse it to avoid heavy reloads.
# Even with files cached on disk, moving weights to RAM is slow.
if "model" not in globals() or model is None:
    print(f"[loader] Loading {model_id} model into RAM.")

    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(model_id)
else:
    print(f"[loader] Reusing {model_id} model that's already in RAM (not reloading).")

# Step 4: Define the API function (a tiny "endpoint")
def run_inference(prompt, max_new_tokens=50):
    """
    Simulated Inference API.
    Think of this like a tiny web endpoint:
      1) It receives the input text (the "request"),
      2) Prepares it for the model (tokenization),
      3) Asks the model to generate a completion (the "response"),
      4) Returns a readable text.
    """

    # Input validation (beginner-friendly guardrails)
    if not isinstance(prompt, str) or len(prompt.strip()) == 0:
        # Friendly error for empty or non-string input
        return "[Error] Please provide a non-empty text prompt."

    # 1. Tokenize the input prompt into model-ready tensors
    # Converts it into token IDs the model can process.
    # Returns a dict of PyTorch tensors (e.g., input_ids, attention_mask).
    inputs = tokenizer(prompt, return_tensors="pt")

    # 2. Generate text with the chosen settings
    # - do_sample=True enables *probabilistic sampling* (randomness). Without this, the model tends to
    #   pick the most likely token each time (greedy decoding), which can be dull/repetitive.
    # - temperature/top_k/top_p only affect results when sampling is enabled.
    # - max_new_tokens controls the *length* of the model's completion (not counting the prompt itself):
    #   smaller = faster/cheaper, larger = more detailed (but slower), Test-Time Compute (TTC)-related
    # - num_return_sequences allows to generate N outputs. When sampling is disabled, can only be set to 1.
    outputs = model.generate(
        **inputs,
        # do_sample=False,
        # temperature=0.7,
        # top_k=50,
        # top_p=0.6,
        max_new_tokens=max_new_tokens,
        # num_return_sequences=1,
    )

    # 3. Convert token IDs back to readable text
    # By default, the first sequence contains: [prompt tokens] + [generated tokens].
    decoded_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # (Optional) Extract only the generated completion (without echoing the prompt)
    # Slice off the first N tokens (the prompt length) to get *only* the new tokens.

    # Find how many tokens belong to the prompt (so we can drop them later)
    # Get the correct key for the token IDs tensor from "inputs".
    prompt_len = inputs["input_ids"].shape[1]

    # Slice the model's output to keep only the newly generated token IDs (drop the prompt part)
    generated_only_ids = outputs[0][prompt_len:]

    # Decode only the generated tokens (no prompt echo) into readable text
    generated_text = tokenizer.decode(generated_only_ids, skip_special_tokens=True)

    # Return both versions to show the difference
    # In a real API, one would usually return just one (often only the generated text).
    return {
        "prompt": prompt,
        "full_output": decoded_text,        # prompt + completion
        "completion": generated_text        # completion only
    }

# Step 5: Try different prompts and simulate real API usage
test_prompts = [
    "Summarize what an LLM does in one sentence.",
    "What's the capital of Brazil?",
]

for prompt in test_prompts:
    print("=" * 60)
    print("💬 Prompt:", prompt)

    result = run_inference(prompt, max_new_tokens=80)  # Simulate an API call

    # Show both views to see what's happening under the hood
    print("🧠 Full Output (echoes prompt):", result["full_output"])
    print("✨ Completion Only:", result["completion"])

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


[loader] Loading unsloth/Llama-3.2-1B model into RAM.
💬 Prompt: Summarize what an LLM does in one sentence.
🧠 Full Output (echoes prompt): Summarize what an LLM does in one sentence. A large language model (LLM) is a computer program that can learn to generate text by analyzing large amounts of data. It can generate text that is coherent, grammatically correct, and semantically meaningful. LLMs are trained on large datasets of text, such as Wikipedia and web pages, and are able to learn from these data to generate text that is similar to the data they have been
✨ Completion Only:  A large language model (LLM) is a computer program that can learn to generate text by analyzing large amounts of data. It can generate text that is coherent, grammatically correct, and semantically meaningful. LLMs are trained on large datasets of text, such as Wikipedia and web pages, and are able to learn from these data to generate text that is similar to the data they have been
💬 Prompt: What's the capita